# pyqplot smoke test

Stage 1 of adding a pyqplot publication-quality plot backend to the MEA pipeline.

**What this notebook does:** verifies that pyqplot installs, runs, and renders every primitive the integration will need (line, scatter, multi-panel, heatmap, colorbar, text, save). Each cell is a single self-contained pass/fail test. No dependency on `engine.py`.

**How to use:** run cells top-to-bottom. Each successful cell prints `PASS: ...`. If any cell raises an exception, **stop** and read the troubleshooting notes below the failing cell. If everything passes, the integration in Stage 2 can proceed.

**About pyqplot:** it is a Python wrapper around a separate C++ `qplot` executable. `pip install pyqplot` installs the Python side; on Windows the C++ binary may need to be downloaded separately from the [qplot releases page](https://github.com/wagenadl/qplot/releases) and added to PATH. If cell 3 (the backend probe) fails with `FileNotFoundError` or a subprocess error, that is the cause.

## 1. Install pyqplot

Idempotent — re-running this cell after a successful install is a no-op.

In [ ]:
%pip install pyqplot

## 2. Import & version

Confirms the Python side is importable. This does NOT yet exercise the C++ backend (that happens in cell 3).

In [ ]:
import qplot as qp
print('qplot module:', qp)
print('qplot version:', getattr(qp, '__version__', '<not exposed>'))
print('PASS: qplot Python module imported')

## 3. Backend probe (the critical test)

Runs the simplest possible plot through to a saved PDF. This is the first cell that invokes the C++ `qplot` binary.

**If this cell fails with `FileNotFoundError`, `WinError 2`, or any "qplot not found"-style error**, the C++ binary is not on PATH. Resolution:
1. Visit https://github.com/wagenadl/qplot/releases
2. Download the latest Windows build (e.g. `qplot-windows-x64-*.zip`)
3. Extract somewhere stable (e.g. `C:\Tools\qplot\`)
4. Add that folder to your Windows PATH (System Properties → Environment Variables)
5. Restart this Jupyter kernel and re-run from cell 1

If the cell succeeds, the printed file path will exist on disk and have non-zero size.

In [ ]:
import os
import tempfile
from pathlib import Path

OUT = Path(tempfile.mkdtemp(prefix='qplot_smoke_'))
print('Smoke-test output dir:', OUT)

qp.figure(str(OUT / 'probe'), 4, 2)
qp.plot([0, 1, 2, 3], [0, 1, 4, 9])
qp.save(str(OUT / 'probe.pdf'))

f = OUT / 'probe.pdf'
assert f.exists(), f'Expected PDF was not created: {f}'
assert f.stat().st_size > 0, f'PDF was created but is empty: {f}'
print(f'PASS: backend probe wrote {f} ({f.stat().st_size} bytes)')

## 4. Inline Jupyter display

Determines whether `qp.figure(...)` + plot commands cause the figure to appear in the notebook output area automatically, or whether we have to manually save and embed.

Two approaches are tried:
- **Approach A:** call `qp.figure()` with no filename, run plot commands, see if anything renders inline.
- **Approach B (fallback):** save to a temporary PNG and embed via `IPython.display.Image`.

Whichever one displays a plot inline determines the strategy Stage 2 will use.

In [ ]:
import numpy as np

# Approach A: implicit display
xx = np.linspace(0, 2 * np.pi, 200)
qp.figure(str(OUT / 'inline_a'), 5, 3)
qp.plot(xx, np.sin(xx))
qp.xaxis('x')
qp.yaxis('sin(x)')
print('Approach A: did a sine wave appear inline above this text? (visual check)')

In [ ]:
# Approach B: explicit PNG + IPython.display
from IPython.display import Image, display

qp.figure(str(OUT / 'inline_b'), 5, 3)
qp.plot(xx, np.cos(xx))
qp.xaxis('x')
qp.yaxis('cos(x)')
qp.save(str(OUT / 'inline_b.png'))
display(Image(filename=str(OUT / 'inline_b.png')))
print('Approach B: a cosine wave should be embedded just above this text')

## 5. Line plot with axis labels & legend

Mirrors what `plot_mea_trace` and most single-panel handlers will need.

In [ ]:
t = np.linspace(0, 100, 2000)  # ms
v = 10 * np.sin(2 * np.pi * 0.05 * t) + np.random.normal(0, 0.5, t.size)

qp.figure(str(OUT / 'cell05_lineplot'), 8, 3)
qp.pen('b')
qp.plot(t, v)
qp.xaxis('Time (ms)')
qp.yaxis('Voltage (uV)')
qp.title('Synthetic MEA-style trace')
qp.save(str(OUT / 'cell05_lineplot.pdf'))
qp.save(str(OUT / 'cell05_lineplot.png'))
display(Image(filename=str(OUT / 'cell05_lineplot.png')))
print('PASS: line plot with labels rendered')

## 6. Multi-line overlay with legend

Mirrors `plot_overlay`. Two traces normalized into the same axes with distinct colors and a legend.

In [ ]:
t = np.linspace(0, 10, 500)
y1 = np.sin(t)
y2 = np.cos(t)

qp.figure(str(OUT / 'cell06_overlay'), 8, 3)
qp.pen('b')
qp.plot(t, y1)
qp.legend('sin')
qp.pen('r')
qp.plot(t, y2)
qp.legend('cos')
qp.xaxis('t')
qp.yaxis('value')
qp.save(str(OUT / 'cell06_overlay.png'))
display(Image(filename=str(OUT / 'cell06_overlay.png')))
print('PASS: two-line overlay rendered (verify legend visually)')

## 7. Scatter / mark

Mirrors `plot_spike_train` (line + spike scatter overlay) and `plot_spike_pca` (PCA scatter).

In [ ]:
rng = np.random.default_rng(0)
n = 80
px = rng.normal(0, 1, n)
py = rng.normal(0, 1, n)

qp.figure(str(OUT / 'cell07_scatter'), 5, 5)
qp.pen('k')
qp.brush('44a')  # bluish fill, qplot 0-9 RGB string
qp.marker('o', 4)
qp.mark(px, py)
qp.xaxis('PC1')
qp.yaxis('PC2')
qp.title('Scatter test')
qp.save(str(OUT / 'cell07_scatter.png'))
display(Image(filename=str(OUT / 'cell07_scatter.png')))
print('PASS: scatter rendered')

## 8. Multi-panel layout (shared x)

Mirrors `plot_ca_trace`, `plot_rt_bank`, `plot_freq_traces`. Two stacked panels sharing the same x-range — pyqplot has no native `sharex`, so we apply matching `qp.xlim` in each panel.

Panel coordinates use the relative form (values <= 1) so they scale with figure size.

In [ ]:
t = np.linspace(0, 20, 400)
raw = np.sin(2 * np.pi * 0.3 * t) + 0.2 * np.random.randn(t.size)
filt = np.sin(2 * np.pi * 0.3 * t)
xlim = (float(t[0]), float(t[-1]))

qp.figure(str(OUT / 'cell08_multipanel'), 8, 5)

# top panel
qp.panel('A', [0.10, 0.10, 0.85, 0.38])
qp.pen('k')
qp.plot(t, raw)
qp.xlim(*xlim)
qp.yaxis('Raw')

# bottom panel
qp.panel('B', [0.10, 0.55, 0.85, 0.38])
qp.pen('b')
qp.plot(t, filt)
qp.xlim(*xlim)
qp.xaxis('Time (s)')
qp.yaxis('Filtered')

qp.save(str(OUT / 'cell08_multipanel.png'))
display(Image(filename=str(OUT / 'cell08_multipanel.png')))
print('PASS: 2-panel layout rendered (verify both panels are visible and aligned)')

## 9. Heatmap with colorbar

Mirrors `plot_spectrogram` (pcolormesh + colorbar) and the spatial scatter heat in `plot_correlation`.

In [ ]:
yy, xx = np.meshgrid(np.linspace(-3, 3, 60), np.linspace(-3, 3, 60), indexing='ij')
zz = np.exp(-(xx**2 + yy**2) / 2) * np.cos(2 * xx)

qp.figure(str(OUT / 'cell09_heatmap'), 6, 5)
qp.luts.set('viridis', 256)
qp.imsc(zz, c0=float(zz.min()), c1=float(zz.max()))
qp.xaxis('X')
qp.yaxis('Y')
qp.cbar(width=10)
qp.caxis()
qp.save(str(OUT / 'cell09_heatmap.png'))
display(Image(filename=str(OUT / 'cell09_heatmap.png')))
print('PASS: heatmap with colorbar rendered (verify colorbar legend is visible)')

## 10. Text annotation

Probes whether `qp.text` can render the params-panel summary at the bottom of a figure. If this works cleanly, Stage 2 can stamp a one-line footer on each PDF; the full multiline parameter dump still goes into a sidecar `.txt` file.

In [ ]:
qp.figure(str(OUT / 'cell10_text'), 6, 4)
qp.plot([0, 1], [0, 1])
qp.xaxis('x')
qp.yaxis('y')
qp.font('Helvetica', 8)
qp.text(0.5, -0.15, 'mea_trace(861).butter_bandpass.sliding_rms  |  window [14487, 16000] ms', align='center')
qp.save(str(OUT / 'cell10_text.png'))
display(Image(filename=str(OUT / 'cell10_text.png')))
print('PASS: text annotation rendered (verify the caption text appears below the axes)')

## 11. PDF and PNG save side-by-side

Final sanity check that both file formats produce reasonable, non-empty output.

In [ ]:
qp.figure(str(OUT / 'cell11_format'), 6, 3)
qp.plot(np.linspace(0, 1, 50), np.sin(np.linspace(0, 6, 50)))
qp.xaxis('x')
qp.yaxis('y')

for ext in ('pdf', 'png', 'svg'):
    p = OUT / f'cell11_format.{ext}'
    qp.save(str(p))
    assert p.exists() and p.stat().st_size > 0, f'failed to write {p}'
    print(f'  wrote {p.name}: {p.stat().st_size} bytes')

print('PASS: PDF, PNG, and SVG all written')

## Summary

If every cell above printed `PASS:` and the inline images look reasonable, Stage 1 is complete and Stage 2 (integration into `engine.py`) can proceed.

**What we need to know going into Stage 2:**
1. Cell 4: did Approach A (implicit display) or only Approach B (explicit `IPython.display.Image`) work? This determines how every Stage 2 handler will end.
2. Cells 5-11: did anything render with obvious visual problems (clipping, missing labels, wrong colors, mangled text)? Note any quirks so they can be fixed once in `qplot_backend.py` rather than repeated across 11 handlers.
3. The output directory printed by cell 3 is where all generated files live — feel free to inspect them directly. The directory can be deleted once you are done.